In [ ]:
import numpy as np
import pandas as pd
import random
#import argparse
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.linear_model import ElasticNet, ElasticNetCV
#from sksurv.linear_model import CoxnetSurvivalAnalysis as CoxPH
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from utils import *
from model_functions import *

random.seed(7)

In [ ]:
from mvdr.ajive.AJIVE import AJIVE

In [ ]:
scaler = StandardScaler()

prots_train_df = pd.read_csv("Data/proteins_metabolites_train.csv", index_col = 'eid')
metabolites_train = scaler.fit_transform(prots_train_df.loc[:, 'Clinical_LDL_C':])
proteins_train =  scaler.fit_transform(prots_train_df.loc[:, : 'ZBTB17'])
prots_val_df = pd.read_csv("Data/proteins_metabolites_set2.csv", index_col = 'eid')
metabolites_set2 =  scaler.fit_transform(prots_val_df.loc[:, 'Clinical_LDL_C':])
proteins_set2 = scaler.fit_transform(prots_val_df.loc[:, : 'ZBTB17'])

In [ ]:
#Initial signal ranks based on PCA
aj10 = AJIVE(init_signal_ranks = [10, 10], center = True).fit([proteins_train, metabolites_train]) 
aj20 = AJIVE(init_signal_ranks = [20, 20], center = True).fit([proteins_train, metabolites_train])

In [ ]:
#aj10.summary()
aj20.summary()

In [ ]:
aj10andersom = AJIVE(init_signal_ranks = [10, 10], center = True).fit([metabolites_train, proteins_train]) 
aj10andersom.summary()

In [ ]:
aj20.common_.view_loadings_[0].shape
aj20.view_specific_[0].individual_.loadings_

In [ ]:
def apply_ajive(train_ajive, protset, metset, eidset):
    # make array
    protset_array = np.array(protset)
    metset_array = np.array(metset)
    
    #Joint loadings
    common_loadings = train_ajive.common_.view_loadings_
    prot_common_loadings = common_loadings[0]
    met_common_loadings = common_loadings[1]
    prot_scores = proteins_train @ prot_common_loadings
    met_scores = metabolites_train @ met_common_loadings
    scores = np.add(prot_scores, met_scores)
    common_ajive = pd.DataFrame(scores * (1/np.linalg.norm(scores, axis=0)), columns = [f'common_comp_{i}' for i in range(common_loadings[0].shape[1])])
    
    #Individual loadings proteins
    view_0_pca = train_ajive.view_specific_[0].individual_
    prot_indiv_loadings_array = np.array(view_0_pca.loadings_) # this are the loadings vectors, you can now manually prject
    #Individual loadings metabolites
    view_1_pca = train_ajive.view_specific_[1].individual_
    met_indiv_loadings_array = np.array(view_1_pca.loadings_) # this are the loadings vectors, you can now manually prject   
    
    #Multiply
    proteins_ind_array = np.dot(protset_array, prot_indiv_loadings_array)
    metabolites_ind_array = np.dot(metset, met_indiv_loadings_array)
    
    # Make dataframe with same names as train
    proteins_ind_ajive = pd.DataFrame(proteins_ind_array, columns=[f'prot_indiv_comp_{i}' for i in range(prot_indiv_loadings_array.shape[1])])
    metabo_ind_ajive = pd.DataFrame(metabolites_ind_array, columns=[f'met_indiv_comp_{i}' for i in range(met_indiv_loadings_array.shape[1])])
    
    ajive_df = pd.concat(
        [pd.DataFrame(eidset.index, columns=['eid']),
         common_ajive,
         proteins_ind_ajive,
         metabo_ind_ajive
        ], 
        axis=1
    )
    return(ajive_df)

In [ ]:
train_df10 = apply_ajive(aj10, proteins_train, metabolites_train, prots_train_df) #to validate function
set2_df10 = apply_ajive(aj10, proteins_set2, metabolites_set2, prots_val_df)
train_df20 = apply_ajive(aj20, proteins_train, metabolites_train, prots_train_df) #to validate function
set2_df20 = apply_ajive(aj20, proteins_set2, metabolites_set2, prots_val_df)

In [ ]:
train_df10

In [ ]:
aj20

In [ ]:
train_df10.to_csv('Data/ajive10_prot_met_train.csv', index = False)
set2_df10.to_csv('Data/ajive10_prot_met_set2.csv', index = False)
train_df20.to_csv('Data/ajive20_prot_met_train.csv', index = False)
set2_df20.to_csv('Data/ajive20_prot_met_set2.csv', index = False)

In [ ]:
def get_data_ajive(config):
    prefix = "Data/Processed/Full"

    normalize = True
    if config['dset'] == "cmb_met_ajive10":
        prots_train_df = pd.read_csv('Data/ajive10_prot_met_train.csv')
        prots_val_df = pd.read_csv('Data/ajive10_prot_met_set2.csv')
    elif config['dset'] == "cmb_met_ajive20":
        prots_train_df = pd.read_csv('Data/ajive20_prot_met_train.csv')
        prots_val_df = pd.read_csv('Data/ajive20_prot_met_set2.csv')
       
    else:
        get_data(config)
    
    
    if config['target'] == "mort":
        target_train = pd.read_csv(prefix + "/mort_full_train.csv", index_col = 'eid')
        target_val = pd.read_csv(prefix + "/mort_full_test.csv", index_col = 'eid')
        #mort_test = pd.read_csv("Data/Processed/Full/mort_full_val.csv", index_col = 'eid')

    elif config['target'] == "frailty":
        target_train = pd.read_csv("Data/frailty_clean_train.csv", index_col = 'eid')
        target_val = pd.read_csv("Data/frailty_clean_set2.csv", index_col = 'eid')


        #add age to dset if we train for frailty
        basicinfo = pd.read_csv("Data/basicinfo_instance_0.csv", index_col = "eid")

        prots_train_df = prots_train_df.merge(basicinfo['age_center.0.0'], on = 'eid')
        prots_val_df = prots_val_df.merge(basicinfo['age_center.0.0'], on = 'eid')



    full_train, train_eids = preprocess(prots_train_df, target_train, target = config['target'] , normalize = normalize)
    full_val, val_eids = preprocess(prots_val_df, target_val, target = config['target'], normalize = normalize)


    return full_train, full_val, train_eids, val_eids
     

In [ ]:
prots_train_df = pd.read_csv('Data/ajive10_prot_met_train.csv', index_col = 'eid')
prots_train_df

In [ ]:
prots_train_df = pd.read_csv('Data/ajive20_prot_met_train.csv')


In [ ]:
prots_train_df.shape

In [ ]:
from utils import get_data

data = get_data({'dset' : "cmb_met_ajive", "target" : "frailty"})

data[0][0].shape

In [ ]:
def find_best_model_modified(dset, target, bootstrap=False):
    # Retrieve data
    trainset, set2, eids_train, eids_set2 = get_data_ajive({'dset': dset, 'target': target})
    trainset_names = get_trainsetnames(dset, target)
    colname = f'en_{target}_{dset}'
     # Get the number of samples from the 'cmb_met' dataset
    cmb_met_data, _, _, _ = get_data({'dset': 'cmb_met', 'target': target})
    num_samples = cmb_met_data[0].shape[0]  # Get the number of participants

    # Initialize storage for results if using bootstrap
    combined_coefs = []
    combined_scores = []
    combined_new_col = []
    combined_r2_or_C = []



    # Perform bootstrap sampling if requested
    if bootstrap:
        for i in range(100):
            # Sample the training data with replacement using the number of samples from 'cmb_met'
            np.random.seed(i)  # Set the seed for reproducibility
            indices = np.random.choice(trainset[0].shape[0], size=num_samples, replace=True)
            sampled_trainset_X = trainset[0][indices]
            sampled_trainset_y = trainset[1][indices]
            if target == 'mort':
                sampled_trainset_event = trainset[2][indices]
                sampled_trainset = (sampled_trainset_X, sampled_trainset_y, sampled_trainset_event)
            else:
                sampled_trainset = (sampled_trainset_X, sampled_trainset_y)
                
            if target == 'frailty':
                best_coefs, scores, R2_save, new_col = find_best_lm_model(sampled_trainset, trainset_names, set2, colname)
                combined_r2_or_C.append(R2_save)
                np.save(f"./output_linear/rsquared/R2_{dset}.npy", R2_save)
            elif target == 'mort':
                best_model, best_coefs, scores, C_save, new_col = find_best_coxph_model(sampled_trainset, trainset_names, set2, colname)
                combined_r2_or_C.append(C_save)
            else:
                raise ValueError(f"Target '{target}' not recognized. Use 'frailty' or 'mort'.")

            combined_coefs.append(best_coefs)
            combined_scores.append(scores)
            combined_new_col.append(new_col)
        
        # Combine the results
        combined_coefs = pd.concat(combined_coefs, axis=1)
        combined_scores = np.array(combined_scores)
        combined_new_col = pd.concat(combined_new_col, axis=1)
        combined_r2_or_C = np.array(combined_r2_or_C)
        np.save(f"./output_linear/bootstrap/{target}/combined_scores_{target}_{dset}.npy", combined_scores)
        combined_coefs.to_csv(f"./output_linear/bootstrap/{target}/coefs_{target}_{dset}.csv")
        combined_new_col.to_csv(f"./output_linear/bootstrap/{target}/en_{target}_{dset}_set2.csv")
        np.save(f"./output_linear/bootstrap/{target}/metric_{target}_{dset}.npy", combined_r2_or_C)


    elif bootstrap == False and target == 'frailty':
        # Find best linear model
        best_coefs, scores, R2_save, new_col = find_best_lm_model(trainset, trainset_names, set2, colname)

        # Save results
        np.save(f"./output_linear/scores_frail/scores_frailty_{dset}.npy", scores)
        best_coefs.to_csv(f'./output_linear/coefs_frail/coefs_frailty_{dset}.csv')
        new_col.to_csv(f'./output_linear/set2frail/en_frailty_{dset}_set2.csv')
        np.save(f"./output_linear/rsquared/R2_{dset}.npy", R2_save)

    elif bootstrap == False and target == 'mort':
        # Find best Cox proportional hazards model
        best_model, best_coefs, scores, C_save, new_col = find_best_coxph_model(trainset, trainset_names, set2, colname)

        # Save results
        np.save(f"./output_linear/scores_mort/scores_mort_{dset}.npy", scores)
        best_coefs.to_csv(f'./output_linear/coefs_mort/coefs_mort_{dset}.csv')
        new_col.to_csv(f'./output_linear/set2mort/en_mort_{dset}_set2.csv')
        np.save(f"./output_linear/concordance/C_mort_{dset}.npy", C_save)
        np.save(f"./output_linear/bestmodel_mort/best_model_mort_{dset}.npy", best_model)

    else:
        raise ValueError(f"Target '{target}' not recognized. Use 'frailty' or 'mort'.")

In [ ]:
find_best_model_modified('cmb_met_ajive10', 'frailty')
find_best_model_modified('cmb_met_ajive20', 'frailty')

In [ ]:
find_best_model_modified('cmb_met_ajive10', 'mort')
find_best_model_modified('cmb_met_ajive20', 'mort')

In [ ]:
find_best_model('cmb_met_pca20', 'frailty')

In [ ]:
find_best_model('cmb_met_pca20', 'mort')